<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;"><img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;"></div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">MSc. CORO DASSIP</div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration — Algorithm</b></h1>
</div>


## Purpose and reading order

This notebook is the **implementation specification** for the corresponding `main.ipynb`.

A competent reader must be able to reconstruct `main.ipynb` from this notebook **without inventing any meaningful implementation decision**. The specification therefore fixes execution order, library operations, helper procedures, stored variables, branches, numerical tolerances, figure construction, filenames, and validation checks.

| Layer | Content | Answers |
| --- | --- | --- |
| **Part I — Complete pseudocode** | Complete coding blueprint, including plotting and validation | Exactly what must be implemented, and in what order? |
| **Part II — Step-by-step elaboration** | Purpose, inputs, procedure, outputs, acceptance contract, and failure mode | Why does each step exist and what must hold before continuing? |

**Standing rule.** Part I is authoritative for implementation order and operations. Part II explains and audits the same workflow and must not introduce behavior absent from `main.ipynb`.

# Part I — Complete Pseudocode

The block below is intentionally detailed. It is a direct implementation specification for reconstructing `main.ipynb` with the same data flow, helper functions, diagnostics, and validation logic.

```text
ALGORITHM Camera_Calibration

  # 0. IMPORTS AND GLOBAL SETTINGS
  IMPORT Path FROM pathlib
  IMPORT cv2
  IMPORT numpy AS np
  IMPORT matplotlib.pyplot AS plt
  CALL np.set_printoptions(precision=6, suppress=True)
  INTERNAL_CORNERS_X ← 8
  INTERNAL_CORNERS_Y ← 6
  PATTERN ← (8, 6)
  N_POINTS ← 48
  SQUARE_SIZE_M ← 0.03
  MIN_VALID_VIEWS ← 3
  DATA_DIR ← Path("../data/calibration_images")
  OUTPUT_DIR ← Path("../outputs/figures")
  CALL OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
  IMAGE_EXTENSIONS ← {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
  REFINE_CRITERIA ← (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

  REQUIRED_OUTPUTS ← [
      "detected_chessboard_corners.png",
      "homography_estimation_pipeline.png",
      "estimated_camera_poses.png",
      "reprojection_results.png",
      "mean_reprojection_error_by_view.png",
      "reprojection_error_distribution.png"
  ]

  # 1. DISCOVER AND LOAD IMAGES
  IF NOT DATA_DIR.is_dir(): RAISE FileNotFoundError
  calibration_image_paths ← sorted supported files in DATA_DIR
  IF calibration_image_paths is empty: RAISE FileNotFoundError
  readable_calibration_views ← []
  FOR image_path IN calibration_image_paths:
      image ← cv2.imread(str(image_path))
      IF image IS None:
          PRINT skipped message
          CONTINUE
      APPEND {path:image_path, name:image_path.name, image:image}
  IF readable_calibration_views is empty: RAISE RuntimeError

  # 2. DETECT AND REFINE CHESSBOARD CORNERS
  FUNCTION detect_and_refine_chessboard_corners(image):
      gray ← cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
      found, corners ← cv2.findChessboardCorners(gray, PATTERN, None)
      IF NOT found OR corners IS None: RETURN None
      refined ← cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), REFINE_CRITERIA)
      RETURN refined.reshape(-1, 2)
  views ← []
  FOR readable_view IN readable_calibration_views:
      detected ← detect_and_refine_chessboard_corners(readable_view["image"])
      IF detected IS None: PRINT skipped message; CONTINUE
      IF detected.shape != (48,2): RAISE ValueError
      view ← COPY(readable_view)
      view["im_pts"] ← detected
      APPEND view TO views
  IF len(views) < MIN_VALID_VIEWS: RAISE RuntimeError

  # 3. BUILD PLANAR WORLD COORDINATES
  points ← np.zeros((48,2), dtype=np.float64)
  index ← 0
  FOR y_index IN range(6):
      FOR x_index IN range(8):
          points[index,0] ← x_index * 0.03
          points[index,1] ← y_index * 0.03
          index ← index + 1
  FOR view IN views: view["plane_pts"] ← points.copy()

  # 4. HOMOGENEOUS COORDINATES AND HARTLEY NORMALIZATION
  FUNCTION to_homogeneous(points):
      homogeneous ← np.ones((points.shape[0],3), dtype=float)
      homogeneous[:,:2] ← np.asarray(points, dtype=float)
      RETURN homogeneous
  FUNCTION normalization_transform(points):
      points ← np.asarray(points, dtype=float)
      centroid ← np.mean(points, axis=0)
      shifted ← points - centroid
      mean_distance ← np.mean(np.linalg.norm(shifted, axis=1))
      IF mean_distance < 1e-12: RAISE ValueError
      scale ← np.sqrt(2.0) / mean_distance
      RETURN np.array([[scale,0,-scale*centroid[0]],[0,scale,-scale*centroid[1]],[0,0,1]], dtype=float)
  FOR view IN views:
      view["T_image"] ← normalization_transform(view["im_pts"])
      view["T_plane"] ← normalization_transform(view["plane_pts"])
      view["normalized_image"] ← (view["T_image"] @ to_homogeneous(view["im_pts"]).T).T
      view["normalized_plane"] ← (view["T_plane"] @ to_homogeneous(view["plane_pts"]).T).T

  # 5. NORMALIZED DLT
  FUNCTION build_homography_dlt_matrix(normalized_plane, normalized_image):
      Q ← np.zeros((2*normalized_image.shape[0], 9), dtype=float)
      FOR i IN range(normalized_image.shape[0]):
          X,Y,_ ← normalized_plane[i]
          u,v,_ ← normalized_image[i]
          Q[2*i] ← [X,Y,1,0,0,0,-u*X,-u*Y,-u]
          Q[2*i+1] ← [0,0,0,X,Y,1,-v*X,-v*Y,-v]
      RETURN Q
  FOR view IN views:
      Q ← build_homography_dlt_matrix(view["normalized_plane"], view["normalized_image"])
      _, sQ, VtQ ← np.linalg.svd(Q)
      h ← VtQ[-1]
      view["Q"] ← Q
      view["h"] ← h
      view["dlt_singular_values"] ← sQ
      view["H_normalized"] ← h.reshape(3,3)

  # 6. DENORMALIZE HOMOGRAPHIES
  FOR view IN views:
      H ← np.linalg.inv(view["T_image"]) @ view["H_normalized"] @ view["T_plane"]
      IF abs(H[2,2]) < 1e-12: RAISE ValueError
      H ← H / H[2,2]
      IF view["im_pts"].shape != (48,2): RAISE ValueError
      IF view["plane_pts"].shape != (48,2): RAISE ValueError
      IF NOT np.all(np.isfinite(H)): RAISE ValueError
      view["H"] ← H

  # 7. ZHANG MULTI-VIEW CONSTRAINTS
  FUNCTION v_ij(hi,hj):
      RETURN np.array([hi[0]*hj[0], hi[0]*hj[1]+hi[1]*hj[0], hi[1]*hj[1], hi[2]*hj[0]+hi[0]*hj[2], hi[2]*hj[1]+hi[1]*hj[2], hi[2]*hj[2]], dtype=float)
  rows ← []
  FOR view IN views:
      h1 ← view["H"][:,0]; h2 ← view["H"][:,1]
      APPEND v_ij(h1,h2) TO rows
      APPEND v_ij(h1,h1)-v_ij(h2,h2) TO rows
  V ← np.vstack(rows)
  _, singular_values_V, Vt_V ← np.linalg.svd(V)
  b ← Vt_V[-1]

  # 8. RECOVER INTRINSICS
  FUNCTION unpack_intrinsic_terms(b):
      b11,b12,b22,b13,b23,b33 ← b
      denominator ← b11*b22 - b12**2
      IF abs(denominator) < 1e-12: RAISE ValueError
      v0 ← (b12*b13 - b11*b23) / denominator
      lambda_ ← b33 - (b13**2 + v0*(b12*b13 - b11*b23)) / b11
      RETURN b11,b12,b22,b13,b23,b33,denominator,v0,lambda_
  b11,b12,b22,b13,b23,b33,denominator,v0,lambda_ ← unpack_intrinsic_terms(b)
  IF lambda_/b11 <= 0 OR lambda_*b11/denominator <= 0:
      b ← -b
      b11,b12,b22,b13,b23,b33,denominator,v0,lambda_ ← unpack_intrinsic_terms(b)
  alpha ← np.sqrt(lambda_/b11)
  beta ← np.sqrt(lambda_*b11/denominator)
  gamma ← -b12*alpha**2*beta/lambda_
  u0 ← gamma*v0/beta - b13*alpha**2/lambda_
  K ← np.array([[alpha,gamma,u0],[0,beta,v0],[0,0,1]], dtype=float)

  # 9. RECOVER CAMERA POSES
  FUNCTION recover_camera_pose(K,H):
      K_inv ← np.linalg.inv(K)
      h1 ← H[:,0]; h2 ← H[:,1]; h3 ← H[:,2]
      lambda_p ← 1.0 / np.linalg.norm(K_inv @ h1)
      r1 ← lambda_p*(K_inv@h1)
      r2 ← lambda_p*(K_inv@h2)
      r3 ← np.cross(r1,r2)
      R_approx ← np.column_stack((r1,r2,r3))
      U,_,Vt ← np.linalg.svd(R_approx)
      R ← U @ Vt
      IF np.linalg.det(R) < 0:
          U[:,-1] ← -U[:,-1]
          R ← U @ Vt
      t ← lambda_p*(K_inv@h3)
      RETURN R,t
  FOR view IN views: view["R"],view["t"] ← recover_camera_pose(K,view["H"])

  # 10. REPROJECT POINTS
  FOR view IN views:
      world_xyz ← np.column_stack((view["plane_pts"], np.zeros(view["plane_pts"].shape[0])))
      camera_frame_points ← world_xyz @ view["R"].T + view["t"]
      projected_h ← camera_frame_points @ K.T
      IF np.any(np.abs(projected_h[:,2]) < 1e-12): RAISE ValueError
      view["world_xyz"] ← world_xyz
      view["projected"] ← projected_h[:,:2] / projected_h[:,2:3]

  # 11. ERRORS AND RMSE
  FOR view IN views:
      view["residuals"] ← view["projected"] - view["im_pts"]
      view["errors"] ← np.linalg.norm(view["residuals"], axis=1)
      view["mean_error"] ← float(np.mean(view["errors"]))
      view["rmse"] ← float(np.sqrt(np.mean(view["errors"]**2)))
  all_reprojection_errors ← np.concatenate([view["errors"] for view in views])
  overall_mean_error ← float(np.mean(all_reprojection_errors))
  overall_rmse ← float(np.sqrt(np.mean(all_reprojection_errors**2)))

  # 12. SIX REQUIRED FIGURES
  # 12.1 Homography pipeline: figsize=(15,3.5), six boxed labels, arrows, dpi=300.
  # 12.2 Mean error by view: figsize=(10,5), bars, dashed overall-mean line, value labels, y-grid.
  # 12.3 Error distribution: figsize=(9,5), histogram bins=20, edgecolor=black, alpha=0.8, mean and RMSE lines.
  # 12.4 Detected corners: 3-column subplot grid, cv2.drawChessboardCorners, BGR→RGB, hide unused axes.
  # 12.5 Camera poses: 3D plot, board at Z=0, camera_center=-R.T@t, label each with path stem.
  # 12.6 Detected vs reprojected: 3-column grid, RGB image, detected marker=o, reprojected marker=x, title includes RMSE to 3 decimals.
  # For every figure: plt.tight_layout(); savefig(..., dpi=300, bbox_inches="tight"); plt.show().

  # 13. FINAL VALIDATION
  IF K.shape != (3,3) OR NOT np.all(np.isfinite(K)): RAISE ValueError
  FOR view IN views:
      IF NOT np.all(np.isfinite(view["H"])): RAISE ValueError
      IF NOT np.allclose(view["R"].T @ view["R"], np.eye(3), atol=1e-6): RAISE ValueError
      IF NOT np.isclose(np.linalg.det(view["R"]), 1.0, atol=1e-6): RAISE ValueError
      IF NOT np.all(np.isfinite(view["errors"])): RAISE ValueError
  missing ← [name for name in REQUIRED_OUTPUTS if not (OUTPUT_DIR/name).exists()]
  IF missing: RAISE FileNotFoundError("Missing outputs: " + ", ".join(missing))
  PRINT "All Camera Calibration validation checks passed."

END ALGORITHM
```


---

# Part II — Step-by-step elaboration

Each step below is the six-section contract corresponding to a numbered section in Part I:

- **Purpose** — why the step exists.
- **Inputs** — artifacts that must already exist.
- **Procedure** — the ordered computation.
- **Produces** — outputs, shapes, or invariants created by the step.
- **Acceptance contract** — conditions that must hold before proceeding.
- **Failure mode** — what failure means and how execution should respond.

**Standing rule.** The acceptance contract of step *k* is the precondition of step *k+1*.

## 0. Start

**Purpose**

Establish the scope and a clean execution order.

**Inputs**

The problem statement, theory, requirements and main notebook in this lab.

**Procedure**

1. Open the lab with the project environment selected in VS Code.
2. Treat this notebook as the coding plan for `main.ipynb`; do not compute results here.
3. Proceed only in the order below because later calibration quantities depend on earlier validated geometry.

**Produces**

Selected environment and a top-to-bottom execution plan.

**Acceptance contract**

Review the local assumptions before using cached notebook outputs.

**Failure mode**

Resolve missing prerequisites first; do not infer a successful run from stored outputs.

## 1. Create the Setup

**Purpose**

Fix the calibration geometry and numerical environment.

**Inputs**

Python environment; 8 by 6 internal corners.

**Procedure**

1. Import `Path`, OpenCV, NumPy, and Matplotlib.
2. Set NumPy display precision so matrices and residuals are readable.
3. Define square size, 8×6 internal corners, minimum valid views, and corner-refinement criteria.

**Produces**

Imports; 0.03 m spacing; minimum 3 views.

**Acceptance contract**

Check the board uses internal corners, not numbers of squares.

**Failure mode**

Stop on missing dependencies or an incorrect board specification.

## 2. Create Repository-Relative Paths

**Purpose**

Keep the experiment portable within the lab.

**Inputs**

Lab notebooks directory.

**Procedure**

1. Create the calibration-image path and `outputs/figures/` path.
2. Create the output directory if needed.
3. Define supported image extensions; use no machine-specific absolute path.

**Produces**

DATA_DIR and OUTPUT_DIR paths.

**Acceptance contract**

Input directory exists; output directory is writable.

**Failure mode**

Resolve the working directory before loading data.

## 3. Discover and Validate Calibration Images

**Purpose**

Build a deterministic image inventory.

**Inputs**

Supported files under data/calibration_images.

**Procedure**

1. Enumerate supported image files and sort them deterministically.
2. If no image is found, stop with `FileNotFoundError`.
3. Create containers for readable images and retained calibration views.

**Produces**

Sorted paths and readable_calibration_views.

**Acceptance contract**

At least one readable image; retain names alongside arrays.

**Failure mode**

Log and skip unreadable files; stop if none remain.

## 4. Detect and Refine Chessboard Corners

**Purpose**

Obtain consistent subpixel correspondences.

**Inputs**

Readable BGR images; REFINE_CRITERIA.

**Procedure**

1. For each image: read it and convert it to grayscale.
2. Detect the complete 8×6 chessboard pattern.
3. If found, refine corners with `cornerSubPix`; otherwise reject that view.
4. Store image name, image array, and refined 2-D points.
5. Stop if fewer than the required number of valid views remain.

**Produces**

views with im_pts shaped (48, 2), in pixels.

**Acceptance contract**

Complete 8 by 6 pattern; at least 3 retained views.

**Failure mode**

Skip incomplete detections explicitly; stop below MIN_VALID_VIEWS.

## 5. Build Metric Planar Coordinates

**Purpose**

Attach metric board coordinates to detected corners.

**Inputs**

Board dimensions and SQUARE_SIZE_M.

**Procedure**

1. Create 48 planar points in the same ordering as detected corners.
2. Set each point to `(x·square_size, y·square_size)` in metres.
3. Keep the board on `Z=0`; add the zero z-coordinate only when 3-D points are needed.

**Produces**

plane_pts shaped (48, 2), in metres.

**Acceptance contract**

x varies fastest, matching corner ordering; board lies on Z=0.

**Failure mode**

Resolve ordering or units before estimating homographies.

## 6. Define Homogeneous-Coordinate Conversion

**Purpose**

Enable homogeneous matrix multiplication.

**Inputs**

Plane/image points shaped (N, 2).

**Procedure**

1. Create a helper that appends a homogeneous coordinate of 1 to every 2-D point.
2. Use it for both plane and image correspondences.
3. Validate the returned shape before matrix computations.

**Produces**

Homogeneous arrays shaped (N, 3).

**Acceptance contract**

Last column equals 1; first two columns preserve coordinates.

**Failure mode**

Do not pass arrays with missing or extra coordinate columns.

## 7. Normalize Plane and Image Points

**Purpose**

Balance coordinate magnitudes before solving DLT.

**Inputs**

48 plane and image points per view.

**Procedure**

1. Compute each point-set centroid and shift to zero mean.
2. Compute mean Euclidean distance from the centroid.
3. Set scale to `sqrt(2)/mean_distance`; reject coincident-point configurations.
4. Build each 3×3 normalization transform and apply it.
5. Store `T_plane`, `T_image`, and normalized point sets per view.

**Produces**

T_plane, T_image and normalized points.

**Acceptance contract**

Centroid approximately zero; mean radius approximately sqrt(2). Current coincidence guard: mean distance below 1e-12.

**Failure mode**

Stop on coincident points; a large normalization scale alone is not degeneracy.

## 8. Build the Normalized DLT System

**Purpose**

Estimate a planar mapping up to scale.

**Inputs**

Normalized correspondences.

**Procedure**

1. For every correspondence add the two standard DLT rows to matrix `Q`.
2. Solve `Qh=0` by SVD.
3. Select the right-singular vector for the smallest singular value.
4. Reshape it into a normalized 3×3 homography.

**Produces**

Q shaped (96, 9); singular values; H_normalized shaped (3, 3).

**Acceptance contract**

Finite Q and homography; use the last right-singular vector. An exact solution is expected to have a near-zero last singular value.

**Failure mode**

Inspect degenerate geometry if the solution is not isolated; do not reject an exact fit because sigma_max/sigma_min is large.

## 9. Denormalize Every Homography

**Purpose**

Express the homography in metres-to-pixels coordinates.

**Inputs**

H_normalized and both normalization matrices.

**Procedure**

1. Compute `H = inv(T_image) @ H_normalized @ T_plane`.
2. Normalize by `H[2,2]`; reject numerically degenerate scale.
3. Store one finite 3×3 homography per retained view.
4. Recheck that each view still contains 48 plane/image correspondences.

**Produces**

One H shaped (3, 3) per view.

**Acceptance contract**

Current helper rejects abs(H[2,2]) below 1e-12 before scaling; verify 48 paired points.

**Failure mode**

Stop on the helper's scale error. This is a chosen normalization limitation, not proof the plane is parallel to the image.

## 10. Build Zhang Multi-View Constraints

**Purpose**

Combine orthogonality constraints from multiple views.

**Inputs**

Retained homographies.

**Procedure**

1. Extract `h1` and `h2` from every homography.
2. Implement the Zhang `v_ij` vector.
3. Stack `v12` and `v11-v22` for all views into matrix `V`.
4. Solve `Vb=0` by SVD and retain the last right-singular vector `b`.

**Produces**

V shaped (2*number_of_views, 6); unit-norm b shaped (6,).

**Acceptance contract**

Inspect the singular spectrum and residual in the same scaling as V. Current implementation stacks unnormalized rows.

**Failure mode**

Do not infer failure from an absolute residual alone or require noisy data to reach machine precision.

## 11. Recover the Intrinsic Matrix

**Purpose**

Recover the camera's intrinsic parameters.

**Inputs**

b in order (B11,B12,B22,B13,B23,B33).

**Procedure**

1. Unpack `b` into the six symmetric-conic coefficients in the order used by `build_zhang_constraint_vector`.
2. Compute `denominator = b11*b22-b12**2`; the current code raises when its absolute value is below `1e-12`.
3. Compute `v0` and `lambda_`, then the two focal-length radicands.
4. Describe the existing sign-retry branch accurately: it negates `b` and recomputes the expressions, but the radicands are mathematically invariant to a global sign flip.
5. Assemble `K` with the closed-form formulas. Review finiteness and positive focal lengths before interpreting the result; the limitations section distinguishes this review from the current automatic checks.

**Produces**

alpha, beta, gamma, u0, v0 and K shaped (3, 3).

**Acceptance contract**

Review finite radicands and positive focal lengths. Existing final check verifies only K shape and finiteness.

**Failure mode**

Stop analysis if K is nonphysical; sign reversal cannot repair negative radicands, whose ratios are sign-invariant. See implementation limits below.

## 12. Recover One Camera Pose per View

**Purpose**

Recover board-to-camera orientation and translation.

**Inputs**

K and each H.

**Procedure**

1. Compute `K^-1 h1`, `K^-1 h2`, and `K^-1 h3`.
2. Use the norm of `K^-1 h1` for the common pose scale.
3. Form `r1`, `r2`, `r3=r1×r2`, and `t`.
4. Project the approximate rotation to the nearest orthonormal matrix with SVD.
5. If `det(R)<0`, correct the sign so `R` is a proper rotation.
6. Store finite `R` and `t`.

**Produces**

R shaped (3, 3); t shaped (3,), in metres.

**Acceptance contract**

Current scale is 1/norm(K^-1 h1); SVD projection enforces a proper rotation. Check orthogonality and determinant near +1.

**Failure mode**

Do not describe t_z>0 as enforced: current code has no cheirality correction.

## 13. Reproject Calibration Points

**Purpose**

Predict image coordinates from the recovered camera.

**Inputs**

K, R, t; board coordinates on Z=0.

**Procedure**

1. Convert planar points to `(X,Y,0)`.
2. Transform them into the camera frame with `R` and `t`.
3. Project with `K` and divide by homogeneous depth.
4. Reject near-zero depth and store the projected 2-D points.

**Produces**

world_xyz shaped (48, 3); projected shaped (48, 2).

**Acceptance contract**

Current guard checks absolute homogeneous depth against 1e-12. Review positive camera-frame depth separately.

**Failure mode**

Stop on near-zero depth; flag negative depths as physically invalid even if division is finite.

## 14. Compute Calibration Errors

**Purpose**

Measure calibration fit in pixel units.

**Inputs**

Observed and projected points with identical ordering.

**Procedure**

1. Compute residual vectors `projected-observed`.
2. Convert residuals to point-wise Euclidean errors.
3. Compute per-view mean error and RMSE.
4. Concatenate all errors and compute overall mean error and RMSE.

**Produces**

Per-point errors; per-view and global mean/RMSE.

**Acceptance contract**

Euclidean error per point; RMSE=sqrt(mean(error**2)); global metrics pool all points.

**Failure mode**

Investigate nonfinite values or systematic residuals; no universal pixel acceptance threshold is implemented.

## 15. Generate the Required Diagnostics

**Purpose**

Reproduce the six committed diagnostic figures from the same arrays used by the numerical calibration.

**Inputs**

`views`, `overall_mean_error`, `overall_rmse`, `all_reprojection_errors`, `OUTPUT_DIR`, and `REQUIRED_OUTPUTS`.

**Procedure**

1. **Homography pipeline** — `(15, 3.5)` figure; six bordered boxes for World points, Image points, Normalize, Build Q, SVD, Denormalize; positions from `np.linspace(0.08, 0.92, 6)`; connect consecutive boxes with arrows.
2. **Mean reprojection error by view** — `(10,5)` bar chart of `view["mean_error"]` against `view["name"]`; dashed horizontal line at `overall_mean_error`; annotate every bar to two decimals; y-grid and legend.
3. **Error distribution** — `(9,5)` histogram of `all_reprojection_errors` using `bins=20`, `edgecolor="black"`, `alpha=0.8`; dashed mean line; dotted RMSE line; y-grid and legend.
4. **Detected corners** — 3-column grid; rows are `ceil(number_of_views/3)`; reshape corners to `(-1,1,2)` and cast to `np.float32`; call `cv2.drawChessboardCorners`; convert BGR to RGB; hide unused axes.
5. **Estimated camera poses** — 3-D figure; plot the board at `Z=0`; compute each camera centre as `-R.T @ t`; label with `view["path"].stem`; axes in metres.
6. **Detected vs reprojected** — same 3-column grid; show RGB image; detected points use `marker="o", s=24`; reprojected points use `marker="x", s=24`; title includes view name and RMSE to three decimals.
7. For every figure call `plt.tight_layout()`, save with `dpi=300` and `bbox_inches="tight"`, then `plt.show()`.

**Produces**

Exactly the six filenames declared in `REQUIRED_OUTPUTS`.

**Acceptance contract**

All figures are generated from the same retained views and residual arrays used by the numerical pipeline; filenames, subplot logic, overlays, labels, and camera-centre equation match `main.ipynb`.

**Failure mode**

A missing figure or a figure generated from different data is an incomplete implementation.


## 16. Run Final Validation

**Purpose**

Replicate the executable notebook's final validation exactly.

**Inputs**

`K`, retained `views`, `REQUIRED_OUTPUTS`, and `OUTPUT_DIR`.

**Procedure**

1. Require `K.shape == (3,3)` and all entries finite.
2. For every view require finite `H`.
3. Require `np.allclose(R.T @ R, np.eye(3), atol=1e-6)`.
4. Require `np.isclose(np.linalg.det(R), 1.0, atol=1e-6)`.
5. Require finite `view["errors"]`.
6. Build `missing = [name for name in REQUIRED_OUTPUTS if not (OUTPUT_DIR / name).exists()]`.
7. Raise `FileNotFoundError("Missing outputs: " + ", ".join(missing))` if any are missing.
8. Otherwise print exactly `All Camera Calibration validation checks passed.`

**Produces**

A deterministic pass/fail outcome matching the current final validation cell.

**Acceptance contract**

All numerical checks pass with the explicit `1e-6` tolerances and all six output files exist.

**Failure mode**

Any failed numerical or artifact check stops execution.
